# Simple Linear Regression II: Python Companion
**ENGR 1451/2451 · Exploratory Data Science · w06b**

The **slides focus on concepts**. This notebook focuses on the Python needed to:

- fit a simple linear regression with `statsmodels`,
- extract the slope, intercept, RMSE, and \(R^2\),
- calculate fitted values and residuals,
- make a few diagnostic plots, and
- refit the model for a simple sensitivity check.

We continue the 2017 state/DC poverty versus HS graduation example from w06a.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import scipy.stats as stats

pd.set_option("display.precision", 3)

counties = pd.read_csv(Path("data/county_complete.csv"))


## 1. Prepare the 2017 state-level data

This is the same aggregation used in w06a. The regression itself gives each state/DC one observation.


In [ ]:
year = "2017"
pop_col = f"pop{year}"
hs_col = f"hs_grad_{year}"
poverty_col = f"poverty_{year}"

county_data = counties[
    ["state", pop_col, hs_col, poverty_col]
].dropna().copy()

county_data["hs_total"] = county_data[hs_col] * county_data[pop_col]
county_data["poverty_total"] = county_data[poverty_col] * county_data[pop_col]

state_data = (
    county_data
    .groupby("state", as_index=False)
    .agg(
        population=(pop_col, "sum"),
        hs_total=("hs_total", "sum"),
        poverty_total=("poverty_total", "sum"),
    )
)

state_data["hs_grad"] = state_data["hs_total"] / state_data["population"]
state_data["poverty"] = state_data["poverty_total"] / state_data["population"]

state_data = state_data[["state", "hs_grad", "poverty"]]
state_data.head()


## 2. Fit the regression with `statsmodels`

Formula syntax is `response ~ predictor`, similar to R's `lm()`.


In [ ]:
fit = smf.ols("poverty ~ hs_grad", data=state_data).fit()

rmse = np.sqrt(np.mean(fit.resid**2))

results = pd.Series({
    "Intercept": fit.params["Intercept"],
    "Slope": fit.params["hs_grad"],
    "RMSE": rmse,
    "R-squared": fit.rsquared,
})

results


In [ ]:
x_grid = np.linspace(state_data["hs_grad"].min(),
                     state_data["hs_grad"].max(), 100)
prediction_data = pd.DataFrame({"hs_grad": x_grid})

plt.figure(figsize=(6.5, 4))
plt.scatter(state_data["hs_grad"], state_data["poverty"], alpha=0.8)
plt.plot(x_grid, fit.predict(prediction_data))
plt.xlabel("HS graduate rate (%)")
plt.ylabel("Poverty rate (%)")
plt.title("2017 state/DC data with least-squares line")
plt.show()


## 3. Work with fitted values and residuals


In [ ]:
state_data = state_data.assign(
    fitted=fit.fittedvalues,
    residual=fit.resid,
)

state_data[["state", "poverty", "fitted", "residual"]].head()


In [ ]:
plt.figure(figsize=(6.5, 4))
plt.scatter(state_data["hs_grad"], state_data["residual"], alpha=0.8)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("HS graduate rate (%)")
plt.ylabel("Residual (percentage points)")
plt.title("Residual plot")
plt.show()


## 4. Two quick shape checks

The slides explain what these displays mean.


In [ ]:
# Histogram
plt.figure(figsize=(6.5, 4))
plt.hist(state_data["residual"], bins="auto", edgecolor="black")
plt.xlabel("Residual (percentage points)")
plt.ylabel("Count")
plt.title("Residual histogram")
plt.show()

# Normal probability (Q-Q) plot
(theoretical, ordered), (slope, intercept, r) = probplot(
    state_data["residual"], dist="norm", fit=True
)

plt.figure(figsize=(6.5, 4))
plt.scatter(theoretical, ordered)

# Reference line
x = np.array([theoretical.min(), theoretical.max()])
plt.plot(x, intercept + slope * x)

plt.xlabel("Normal-reference quantile")
plt.ylabel("Ordered residual (percentage points)")
plt.title("Normal probability plot")
plt.show()


## 5. Simple sensitivity check

Refit after removing one observation and compare the fitted coefficients.


In [ ]:
state_to_check = state_data.loc[
    state_data["residual"].abs().idxmax(), "state"
]

reduced_data = state_data.loc[state_data["state"] != state_to_check]
fit_without = smf.ols("poverty ~ hs_grad", data=reduced_data).fit()

comparison = pd.DataFrame({
    "All states/DC": fit.params,
    f"Without {state_to_check}": fit_without.params,
})

comparison


## 6. Practice

Use the fitted model and the objects already created above.

1. Predict the poverty rate when `hs_grad = 90`.
2. Find the five states/DC with the largest absolute residuals.
3. Change `state_to_check` to another state and compare the new slope with the original slope.

### Core Python patterns

| Task | Python |
|---|---|
| Fit the model | `smf.ols("poverty ~ hs_grad", data=state_data).fit()` |
| Coefficients | `fit.params` |
| Fitted values | `fit.fittedvalues` |
| Residuals | `fit.resid` |
| \(R^2\) | `fit.rsquared` |
| Predict | `fit.predict(new_data)` |
